In [ ]:
import os
import json
#pandas -framework allows to work with tabular data like excel /csv we see rows and cols sructured table..to access specific row
import pandas as pd
from dotenv import load_dotenv

#---step2: load .env---
load_dotenv()

#---step3 : Load config.json---

with open ("config/config.json","r") as f:
#inside this config variable it loads everything from config.json
    config = json.load(f)


#step 4: 
#Load transcripts

df =pd.read_csv("data/transcripts.csv")

print("Data Loaded")
print(df.head(),"\n")

#step 5 :Initialize LLM
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI

print("LLM_PROVIDER from env:", os.getenv("LLM_PROVIDER"))

def load_llm(config):
    provider = os.getenv("LLM_PROVIDER")

    if provider =="openai":
        llm = ChatOpenAI(

            model = config["llm"]["openai_model"],
            temperature = config["llm"]["temperature"]

        )

    elif provider == "gemini":
        llm = ChatGoogleGenerativeAI(
            model=config["llm"]["gemini_model"],
            temperature=config["llm"]["temperature"]
        )

    else:
        raise ValueError("Invalid LLM_PROVIDER")

    return llm

llm = load_llm(config)
print(f"✅ LLM Initialized using provider: {os.getenv('LLM_PROVIDER')}")

# --- Step 6: Test LLM ---
response = llm.invoke("Say 'setup successful' in one short sentence.")
print("\n🧠 LLM Response:")
print(response.content)

In [ ]:
#----Step 1: Imports (safe to repeat)---

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel,Field

#--Step2 : Define Output Schema---
"""
This defines the shape you want the LLM's answer forced into: a call_type string and a confidence float.
 The description in each Field isn't just documentation — it actually gets included in the instructions sent to the LLM 
 (you'll see this in the next step), guiding it on what to put in each field.
class ClassificationOutput(BaseModel):
    call_type: str
    confidence: float
This is also completly valid The only difference from the version with Field(...) is that you lose the description metadata:
"""
class ClassificationOutput(BaseModel):
    call_type: str = Field(description = "Type of customer call")
    confidence: float = Field(description ="Confidence score between 0 and 1")

"""
This wraps your schema into a parser object. It does two jobs:

Generates format instructions (a text block telling the LLM exactly what JSON structure to return)
Later, takes the LLM's raw text output and converts it into a validated ClassificationOutput object

"""
parser = PydanticOutputParser(pydantic_object = ClassificationOutput )


#---Step 3 --Prompt template
"""
{transcript}, {labels}, {format_instructions} are placeholders inside the template string
input_variables=["transcript"] — this is the only value you'll supply fresh each time you call the chain (the actual customer transcript text)
partial_variables — these two are filled in once, right now, not at call-time:
format_instructions comes from parser.get_format_instructions() — this asks the parser to generate that JSON-format instruction text automatically, based on your ClassificationOutput schema
labels pulls directly from your config.json — ["billing", "claims", "complaint", "general_query"] — so the LLM knows the valid categories to choose from

"""
prompt = PromptTemplate(
    template= """

    You are a call classification assistant.
    Classify the following customer support transcript into one of these categories:
    {labels}

    Transcript:
    {transcript}

    {format_instructions}

    """,

    input_variables = ["transcript"],
    partial_variables = {
        "format_instructions": parser.get_format_instructions(),
        "labels": config["classification"]["labels"]
    }


)

#--Step 4 : Create chain
"""
transcript text flows into prompt (fills the template) → 
llm (generates a raw text response, hopefully valid JSON) → 
parser (converts that JSON text into a real ClassificationOutput object, validating it along the way).
"""
classification_chain = prompt | llm | parser

#--Step 5: Test with one transcript----

#indexLocation 0 fetch the first record and transcript column
""" 
df.iloc[0]["transcript"] — grabs row 0's value from the transcript column of your CSV (loaded earlier via pandas)
.invoke({"transcript": sample_text}) — only need to pass transcript here, since labels and format_instructions were already locked in as partials in Step 4
result will be an actual ClassificationOutput instance — so you could access result.call_type and result.confidence directly, not just a raw dict




"""


sample_text = df.iloc[0]["transcript"]


result = classification_chain.invoke(
    {
        "transcript" : sample_text
    }
)

print("Classification Result:")
print(result)















In [ ]:
# This is to get call_type and confidence for all the records
#tqdm is the library allows us to see the progress in progress bar

from tqdm import tqdm

results = []

for i,row in tqdm(df.iterrows(),total = len(df),desc = "Classifying calls"):
    try:
        output = classification_chain.invoke({
            "transcript" : row["transcript"]
            })
        results.append({
            "call_id" :  row["call_id"],
            "predicted_call_type" : output.call_type,
            "confidence" : output.confidence

        })
    except Exception as e:
        print(f"❌ Error at row {i} {e}")
        results.append({
            "call_id" :  row["call_id"],
            "predicted_call_type" : None,
            "confidence" : None
        })
    

results_df = pd.DataFrame(results)
df = df.merge(results_df,on = "call_id")

print("\n ☑️Batch classification completed\n")

print(df[["call_id","expected_call_type","predicted_call_type","confidence"]])


In [ ]:
# Step 1: Define routing logic:
def route_call(call_type):
    if call_type == "billing":
        return ["Knowledge_accuracy","resolution_quality"]
    elif call_type == "claims":
        return ["Knowledge_accuracy","resolution_quality"]
    elif   call_type == "complaint":
        return["tone_empathy","resolution_quality"]
    elif  call_type == "general_query":
        return["Knowledge_accuracy"]
    else:
        return["knowledge_accuracy"] #fallback

#---Step 2 Apply routing to dataset
#on the column predicted_call_type apply route_call and what route_call returns save it in new column evaluation_plan

df["evaluation_plan"] = df["predicted_call_type"].apply(route_call)

print("☑️Routing Applied\n")

print(df[["call_id","predicted_call_type","evaluation_plan"]])

          

In [ ]:
#Step1:imports


from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel,Field

#step2:Define output schema---


class ToneEvaluation(BaseModel):
    score:int = Field(description="Score between 1 and 5")
    reasoning: str = Field(description= "Explanation of the score")

tone_parser = PydanticOutputParser(pydantic_object= ToneEvaluation)

#Step 3--- Prompt template

tone_prompt = PromptTemplate(

    template= """ 
    You are a QA evaluator for customer support calls.
    Evaluate the agent's tone and empathy in the following transcript.

    Consider:
    - Did the agent acknowledge the customer's issue?
    -Was the tone polite and professional?
    -Did the agent show empathy?

    Transcript:
    {transcript}

    {format_instructions}
    """,

    input_variables = ["transcript"],
    partial_variables= {
         "format_instructions": tone_parser.get_format_instructions()

    }
)

tone_chain = tone_prompt | llm | tone_parser

#--step 5:Test on ONE complaint call(important)
""" 
This is the filter step. df["predicted_call_type"]=="complaint" creates a Series of True/False values (one per row) —
 True where the predicted type is "complaint". Wrapping that back in df[...] uses that True/False mask to keep only the rows where it's True.

"""
sample_text = df[df["predicted_call_type"]=="complaint"].iloc[0]["transcript"]

result = tone_chain.invoke({
    "transcript" :  sample_text
})

print("Tone evaluation result")
print(result)


In [ ]:
#Step1:imports


from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel,Field

#step2:Define output schema---


class ResolutionEvaluation(BaseModel):
    score:int = Field(description="Score between 1 and 5")
    reasoning: str = Field(description= "Explanation of the score")

resolution_parser = PydanticOutputParser(pydantic_object= ResolutionEvaluation)

#Step 3--- Prompt template

resolution_prompt = PromptTemplate(

    template= """ 
    You are a QA evaluator for customer support calls.
    Evaluate the resolution quality of the agent.

    Consider:
    - Did the agent fully resolve the customer's issue?
    -Were next steps clearly communicated?
    -Did the agent confirm resolution before ending?

    Transcript:
    {transcript}

    {format_instructions}
    """,

    input_variables = ["transcript"],
    partial_variables= {
         "format_instructions": resolution_parser.get_format_instructions()

    }
)

resolution_chain = resolution_prompt | llm | resolution_parser

#--step 5:Test on ONE complaint call(important)
""" 
This is the filter step. df["predicted_call_type"]=="complaint" creates a Series of True/False values (one per row) —
 True where the predicted type is "complaint". Wrapping that back in df[...] uses that True/False mask to keep only the rows where it's True.

"""
sample_text = df[df["predicted_call_type"]=="complaint"].iloc[0]["transcript"]

result = resolution_chain.invoke({
    "transcript" :  sample_text
})

print("Resolution evaluation result")
print(result)


In [ ]:
#Step1:imports


from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel,Field

#step2:Define output schema---


class KnowledgeEvaluation(BaseModel):
    score:int = Field(description="Score between 1 and 5")
    reasoning: str = Field(description= "Explanation of the score")

Knowledge_parser = PydanticOutputParser(pydantic_object= KnowledgeEvaluation)

#Step 3--- Prompt template

Knowledge_prompt = PromptTemplate(

    template= """ 
    You are a QA evaluator for customer support calls.
    Evaluate the agent's knowlege and accuracy in the following transcript.

    Consider:
    - Did the agent provide correct and relevant information?
    -Was the explanation clear and easy to unserstand?
    -Did the agent avoid vague or misleading statement?

    Transcript:
    {transcript}

    {format_instructions}
    """,

    input_variables = ["transcript"],
    partial_variables= {
         "format_instructions": Knowledge_parser.get_format_instructions()

    }
)

knowledge_chain = Knowledge_prompt | llm | Knowledge_parser

#--step 5:Test on ONE complaint call(important)
""" 
This is the filter step. df["predicted_call_type"]=="complaint" creates a Series of True/False values (one per row) —
 True where the predicted type is "complaint". Wrapping that back in df[...] uses that True/False mask to keep only the rows where it's True.

"""
sample_text = df[df["predicted_call_type"]=="complaint"].iloc[0]["transcript"]

result = knowledge_chain.invoke({
    "transcript" :  sample_text
})

print("Knowledge evaluation result")
print(result)


In [ ]:
from tqdm import tqdm

#..step 1--Evalutaion runner

def run_evaluations(transcript,eval_plan):
    results = {}
    if "tone_empathy" in eval_plan:
        try:
            tone_result = tone_chain.invoke({"transcript": transcript})
            # results with key "tone" is fillled with the output from tone_result
            results["tone"] = tone_result.model_dump()
        except Exception as e:
            results["tone"] = {"error" : str(e)}

    if "knowledge_accuracy" in eval_plan:
            try:
                knowledge_result = knowledge_chain.invoke({"transcript": transcript})
                results["knowledge"] = knowledge_result.model_dump()
            except Exception as e:
                results["knowledge"] = {"error" : str(e)}


    if "resolution_quality" in eval_plan:
            try:
                resolution_result = resolution_chain.invoke({"transcript": transcript})
                results["resolution"] = resolution_result.model_dump()
            except Exception as e:
                results["resolution"] = {"error" : str(e)}
    

    return results

#Step2: Applyn to entire datset

evaluation_output = []

for i, row in tqdm(df.iterrows(),total=len(df),desc ="Running Evaluations"):
    output = run_evaluations(row["transcript"],row["evaluation_plan"])

    evaluation_output.append({
         "call_id": row["call_id"],
         "evaluation_output" : output
    })

#Convert to DataFrame

eval_df = pd.DataFrame(evaluation_output)

#Merge

df = df.merge(eval_df,on="call_id")

print("\n☑️Evaluation Completed\n")


import pprint
pprint.pprint(df.iloc[0]["evaluation_output"])                       











    

In [ ]:
#--Step1---Schema

class FinalReport(BaseModel):
    summary: str = Field(description="Overall evaluation summary")
    recommendation:list[str] = Field(description ="List of actionable improvements")

final_parser = PydanticOutputParser(pydantic_object= FinalReport)

#Step 2--prompt
final_prompt= PromptTemplate(

    template=""" 
You are a QA manager reviewing customer support calls.
Based on the evaluation results below,generate:

1.A consics summary of the agent's performance
2.A list of actionable recommendations for improvement

Evaluation Data:

{evaluation_output}

IMPORTANT:
-Be specific and practical
-Do not repeat scores
-Focus on improvement

{format_instructions}
""",

    input_variables=["evaluation_output"],
    partial_variables={
        "format_instructions": final_parser.get_format_instructions()
        })



#--Step3 : Chain---

final_chain = final_prompt |llm|final_parser

#--Step 4: Test on one row

sample_eval = df.iloc[0]["evaluation_output"]

result = final_chain.invoke({
    "evaluation_output" : sample_eval
})

print("☑️Final QA Report")

print(result)
























In [ ]:
from tqdm import tqdm
final_output = []

for i, row in tqdm(df.iterrows(),total=len(df),desc ="Generating Final Reports"):
    try:
        result = final_chain.invoke({
            "evaluation_output" : row["evaluation_output"]
        })
        final_output.append({
            "call_id": row["call_id"],
            "summary" : result.summary,
            "recommendation" : result.recommendation
        })
    except Exception as e:
        print(f"❌ Error ar row{i}: {e}")
        final_output.append({
            "call_id" : row["call_id"],
            "summary":None,
            "recommendation" :None
        })

#Convert to DataFrame

final_df = pd.DataFrame(final_output)

#Merge

df = df.merge(final_df,on="call_id")

print("\n☑️Final Report Generation\n")


#Show Fianl result

df[[
    "call_id",
    "predicted_call_type",
    "evaluation_output",
    "summary",
    "recommendation"
]]      


In [ ]:
df.to_excel("data/output.xlsx",index=False)
print("☑️Results saved to data/output.xlsx")
